# Energy Indicators, Data Retrieval (earthkit.data)

Retrieves real Climate DT projection data (SSP3-7.0, IFS-NEMO) for
the `energy_indicators` Basic and Advanced tutorials, using
`earthkit.data`, ECMWF's official external-user interface, via the
Polytope engine (MN5 databridge). Requires a valid DESP/Polytope
authentication token (see `desp-authentication.py`).

For background on the Polytope/DESP access mechanisms and further
retrieval examples, see
[polytope-examples](https://github.com/destination-earth-digital-twins/polytope-examples/tree/main).

**Requirements**: this notebook is standalone and does not depend on
`energy_indicators` itself. Beyond a standard Python 3.11 scientific
stack (`xarray`, `netCDF4`, `numpy`, `scipy`, `matplotlib`), it needs
`earthkit-data`, `healpy`, and `cartopy`; `desp-authentication.py`
additionally needs `conflator` and `lxml`. All of these are covered
by the `polytope-examples` environment linked above.

**Sites**, a real, coherent regional story on each side:
- Moray East (Scotland, offshore wind, 58.167N, -2.699W)
- Tico (Villar de los Navarros, Zaragoza, Spain, onshore wind,
  41.159N, -1.044W)
- Zaragoza (city, 41.656N, -0.877W), real regional capital, used for
  the Demand story
- Fuendetodos (solar farm, Villanueva de Huerva, Zaragoza, Spain,
  41.200N, -1.283W), used for the Solar story

**Variables and period**: 100 m wind (`u`, `v`) at both wind sites,
for January 2035 and, for Tico only, July 2035, supporting a
winter/summer PV potential comparison at Fuendetodos (Fuendetodos and
Tico are ~20 km apart, so Tico's wind is used as a stand-in); 2 m
temperature (`2t`) and solar radiation (`avg_sdswrf`) at Zaragoza and
Fuendetodos, for both months.

**Resolution**: `resolution="standard"` (HEALPix H128, ~50 km cells)
is used deliberately throughout, to keep this notebook runnable on a
regular laptop rather than requiring HPC access. Climate DT itself
natively supports much finer resolution (`resolution="high"`,
HEALPix H1024, ~5-10 km). Each site's data is the nearest available 
`standard`-resolution cell rather than its exact coordinate.

**Retrieval strategy**: each distinct (variable, level, month)
combination is retrieved once, globally, split into ~week-sized
chunks that are cached to disk as they complete, and every site that
needs that data is extracted from the same retrieval rather than
fetched separately. Chunking keeps peak memory manageable on a
regular laptop, since decoding a full month's global field in one
operation is heavy.

**Outputs**: per-site time series (`data/<site>_<variable>_<year><month>.nc`),
a global monthly-mean temperature map, and a European context map,
see the summary table at the end. Large intermediate files
(`data/_*.nc`, `data/tmp/`) are retrieval scratch and can be deleted
after.

## Authentication

In [ ]:
%%capture cap
%run ./desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]


## Imports

In [ ]:
import os
import gc
import calendar
import numpy as np
import xarray as xr
import healpy as hp
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata

import earthkit.data


## Locations and Period

In [ ]:
sites = {
    "moray_east": {"name": "Moray East", "lat": 58.167, "lon": -2.699, "type": "offshore wind"},
    "tico": {"name": "Tico", "lat": 41.159, "lon": -1.044, "type": "onshore wind"},
    "zaragoza": {"name": "Zaragoza", "lat": 41.656, "lon": -0.877, "type": "city (demand)"},
    "fuendetodos": {"name": "Fuendetodos", "lat": 41.200, "lon": -1.283, "type": "solar"},
}

YEAR = "2035"
WINTER_MONTH = "01"
SUMMER_MONTH = "07"

NSIDE = 128  # resolution="standard" (H128); would be 1024 for "high"

os.makedirs("data", exist_ok=True)
os.makedirs("plots", exist_ok=True)


## Helpers

In [ ]:
def retrieve_raw(levtype, param, date, time="0000/to/2300/by/0100", levelist=None,
                  stream="clte", resolution="standard"):
    request = {
        "class": "d1",
        "dataset": "climate-dt",
        "experiment": "ssp3-7.0",
        "activity": "projections",
        "model": "ifs-nemo",
        "generation": "2",
        "realization": "1",
        "resolution": resolution,
        "stream": stream,
        "type": "fc",
        "expver": "0001",
        "levtype": levtype,
        "param": param,
    }
    if date is not None:
        request["date"] = date
        request["time"] = time
    if levelist is not None:
        request["levelist"] = levelist

    raw = earthkit.data.from_source(
        "polytope", "destination-earth", request,
        address="polytope.mn5.apps.dte.destination-earth.eu", stream=False,
    )
    return raw.to_xarray()


def strip_earthkit_attrs(ds):
    ds = ds.copy()
    ds.attrs.pop("_earthkit", None)
    for var in ds.variables.values():
        var.attrs.pop("_earthkit", None)
    return ds


def attach_latlon(ds, nside=NSIDE):
    if "latitude" in ds.coords and "longitude" in ds.coords:
        return ds.rename({"latitude": "lat", "longitude": "lon"})
    if "lon" in ds.coords and "lat" in ds.coords:
        return ds

    cell_dim = None
    for candidate in ("values", "cell", "ncells", "rgrid"):
        if candidate in ds.dims:
            cell_dim = candidate
            break
    if cell_dim is None:
        raise ValueError(f"Could not find a HEALPix cell dimension among {list(ds.dims)}")

    cell_ids = np.arange(ds.sizes[cell_dim])
    theta, phi = hp.pix2ang(nside, cell_ids, nest=True)
    lat_vals = 90.0 - np.degrees(theta)
    lon_vals = np.degrees(phi)

    return ds.assign_coords({"lat": (cell_dim, lat_vals), "lon": (cell_dim, lon_vals)})


def select_nearest(ds, lat, lon):
    lons = ds["lon"].values
    lats = ds["lat"].values
    dist2 = (lats - lat) ** 2 + (lons - lon) ** 2
    idx = int(np.argmin(dist2))
    cell_dim = ds["lon"].dims[0]
    return ds.isel({cell_dim: idx})


def week_chunks(year, month, chunk_days=7):
    n_days = calendar.monthrange(int(year), int(month))[1]
    chunks = []
    start = 1
    while start <= n_days:
        end = min(start + chunk_days - 1, n_days)
        chunks.append((start, end))
        start = end + 1
    return chunks


def retrieve_chunked(levtype, param, year, month, levelist=None, chunk_days=7, cache_prefix="chunk"):
    chunks_data = []
    for start, end in week_chunks(year, month, chunk_days):
        date_range = f"{year}{month}{start:02d}/to/{year}{month}{end:02d}"
        chunk_file = f"data/_{cache_prefix}_{year}{month}_{start:02d}-{end:02d}.nc"

        if os.path.exists(chunk_file):
            chunk_ds = xr.open_dataset(chunk_file)
            print(f"  {date_range}: loaded existing {chunk_file}")
        else:
            chunk_ds = strip_earthkit_attrs(attach_latlon(
                retrieve_raw(levtype=levtype, param=param, date=date_range, levelist=levelist)
            ))
            chunk_ds.to_netcdf(chunk_file)
            print(f"  {date_range}: retrieved and cached -> {chunk_file}")

        chunks_data.append(chunk_ds)

    combined = xr.concat(chunks_data, dim="forecast_reference_time")
    if "forecast_reference_time" in combined.dims and "time" not in combined.dims:
        combined = combined.rename({"forecast_reference_time": "time"})
    return combined


def normalize_lon(ds):
    ds = ds.copy()
    ds["lon"] = ((ds["lon"] + 180) % 360) - 180
    return ds


def free(*names):
    """Delete large intermediate variables from the notebook's
    namespace once they're no longer needed, and force garbage
    collection so the memory is actually released."""
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()


## Wind, 100 m u/v, both sites, winter

Retrieved once, globally, for January. Both Moray East and Tico are
extracted from the combined result.

In [ ]:
wind_global_winter = normalize_lon(retrieve_chunked(
    levtype="hl", param=["u", "v"], year=YEAR, month=WINTER_MONTH,
    levelist="100", cache_prefix="wind",
))
wind_global_winter


In [ ]:
wind_data = {}
for key in ["moray_east", "tico"]:
    s = sites[key]
    data_file = f"data/{key}_wind_{YEAR}{WINTER_MONTH}.nc"
    if os.path.exists(data_file):
        wind_data[key] = xr.open_dataset(data_file)
        print(f"{s['name']}: loaded existing {data_file}")
        continue
    site_ds = select_nearest(wind_global_winter, s["lat"], s["lon"])
    site_ds.to_netcdf(data_file)
    wind_data[key] = site_ds
    print(f"{s['name']}: extracted -> {data_file} "
          f"(nearest cell at lat={float(site_ds['lat']):.3f}, lon={float(site_ds['lon']):.3f})")

free("wind_global_winter")


## Wind, 100 m u/v, Tico, summer

In [ ]:
tico_summer_file = f"data/tico_wind_{YEAR}{SUMMER_MONTH}.nc"
if os.path.exists(tico_summer_file):
    tico_wind_summer = xr.open_dataset(tico_summer_file)
    print(f"Tico (summer): loaded existing {tico_summer_file}")
else:
    wind_global_summer = normalize_lon(retrieve_chunked(
        levtype="hl", param=["u", "v"], year=YEAR, month=SUMMER_MONTH,
        levelist="100", cache_prefix="wind",
    ))
    tico_wind_summer = select_nearest(wind_global_summer, sites["tico"]["lat"], sites["tico"]["lon"])
    tico_wind_summer.to_netcdf(tico_summer_file)
    print(f"Tico (summer): extracted -> {tico_summer_file}")
    free("wind_global_summer")


## Temperature and Solar Radiation, Zaragoza and Fuendetodos, winter

In [ ]:
sfc_global_winter = normalize_lon(retrieve_chunked(
    levtype="sfc", param=["2t", "avg_sdswrf"], year=YEAR, month=WINTER_MONTH,
    cache_prefix="sfc",
))
sfc_global_winter


In [ ]:
zaragoza_temp = select_nearest(sfc_global_winter, sites["zaragoza"]["lat"], sites["zaragoza"]["lon"])
zaragoza_temp.to_netcdf(f"data/zaragoza_temp_{YEAR}{WINTER_MONTH}.nc")

fuendetodos_winter = select_nearest(sfc_global_winter, sites["fuendetodos"]["lat"], sites["fuendetodos"]["lon"])
fuendetodos_winter.to_netcdf(f"data/fuendetodos_solar_{YEAR}{WINTER_MONTH}.nc")

print("Zaragoza and Fuendetodos (winter) extracted.")


## Temperature and Solar Radiation, Fuendetodos, summer

In [ ]:
sfc_global_summer = normalize_lon(retrieve_chunked(
    levtype="sfc", param=["2t", "avg_sdswrf"], year=YEAR, month=SUMMER_MONTH,
    cache_prefix="sfc",
))

fuendetodos_summer = select_nearest(sfc_global_summer, sites["fuendetodos"]["lat"], sites["fuendetodos"]["lon"])
fuendetodos_summer.to_netcdf(f"data/fuendetodos_solar_{YEAR}{SUMMER_MONTH}.nc")
print("Fuendetodos (summer) extracted.")

free("sfc_global_summer")


## Global map, monthly-mean 2 m temperature

Computed from the already-retrieved winter data above, so no new
request is needed.

In [ ]:
da_global_mean = sfc_global_winter["2t"].mean(dim="time")

global_mean_file = f"data/global_2t_monthly_{YEAR}{WINTER_MONTH}.nc"
xr.Dataset({"2t": da_global_mean}).to_netcdf(global_mean_file)
print(f"computed global monthly mean -> {global_mean_file}")


In [ ]:
lons = da_global_mean["lon"].values
lats = da_global_mean["lat"].values
values = da_global_mean.values.ravel()

grid_deg = 1.0
lon_new = np.arange(-180, 180, grid_deg)
lat_new = np.arange(-90, 90, grid_deg)
lon_grid, lat_grid = np.meshgrid(lon_new, lat_new)
values_grid = griddata(points=(lons, lats), values=values, xi=(lon_grid, lat_grid), method="nearest")

fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"projection": ccrs.PlateCarree()})
cs = ax.pcolormesh(lon_grid, lat_grid, values_grid, transform=ccrs.PlateCarree(),
                    cmap="RdYlBu_r", shading="auto")
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle="--")
plt.colorbar(cs, ax=ax, label="2 m Temperature (K)", orientation="horizontal", pad=0.08, shrink=0.7)
ax.set_title(f"Global mean 2 m Temperature, {YEAR}-{WINTER_MONTH} (monthly mean, SSP3-7.0)")
plt.tight_layout()
plt.savefig(f"plots/global_2t_{YEAR}{WINTER_MONTH}.png", dpi=150, bbox_inches="tight")
plt.show()


## European context map

Marks all four sites. Labels use a fixed offset with an arrow back to
each point, so they stay readable even where sites are close together
(Tico, Zaragoza, and Fuendetodos are all within about 50 km of each
other).

In [ ]:
da_full = sfc_global_winter["2t"].isel(time=0).squeeze()
lons = sfc_global_winter["lon"].values
lats = sfc_global_winter["lat"].values

n, w, s, e = 72, -15, 35, 32  # N, W, S, E - covers UK/Scandinavia to Spain
mask = (lats <= n) & (lats >= s) & (lons >= w) & (lons <= e)

lons_eu = lons[mask]
lats_eu = lats[mask]
values_eu = da_full.values.ravel()[mask]

grid_deg = 0.25
lon_new = np.arange(w, e, grid_deg)
lat_new = np.arange(s, n, grid_deg)
lon_grid, lat_grid = np.meshgrid(lon_new, lat_new)
values_grid = griddata(points=(lons_eu, lats_eu), values=values_eu, xi=(lon_grid, lat_grid), method="linear")

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
cs = ax.pcolormesh(lon_grid, lat_grid, values_grid, transform=ccrs.PlateCarree(),
                    cmap="RdYlBu_r", shading="auto")
ax.add_feature(cfeature.COASTLINE, linewidth=0.75)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle="--")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

label_offsets = {
    "moray_east": (30, 20), "tico": (-60, -35),
    "zaragoza": (50, 20), "fuendetodos": (-70, 20),
}
for key, site in sites.items():
    ax.plot(site["lon"], site["lat"], marker="o", color="black", markersize=8,
             transform=ccrs.PlateCarree())
    ax.annotate(
        f"{site['name']}\n({site['type']})",
        xy=(site["lon"], site["lat"]), xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),
        xytext=label_offsets[key], textcoords="offset points",
        fontsize=9, ha="center",
        arrowprops=dict(arrowstyle="->", color="black", lw=1),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.85),
    )

plt.colorbar(cs, ax=ax, label="2 m Temperature (K)", orientation="horizontal", pad=0.05, shrink=0.7)
ax.set_title(f"Europe, 2 m Temperature context ({YEAR}-{WINTER_MONTH}, first timestep)")
plt.tight_layout()
plt.savefig(f"plots/europe_context_marked_{YEAR}{WINTER_MONTH}.png", dpi=150, bbox_inches="tight")
plt.show()

xr.Dataset({"2t": da_full}).to_netcdf(f"data/europe_2t_{YEAR}{WINTER_MONTH}.nc")

free("sfc_global_winter", "da_global_mean", "da_full")


## Cleanup

Remove the large intermediate chunk files once the per-site files
above are confirmed correct. They aren't needed by the tutorials
directly and shouldn't be committed alongside them.

In [ ]:
# Uncomment to delete the intermediate files once the per-site files
# above are confirmed correct:
#
import glob
for f in glob.glob("data/_wind_*.nc") + glob.glob("data/_sfc_*.nc"):
    os.remove(f)
    print(f"removed {f}")


## Summary, files produced

| File | Contents | Feeds |
|---|---|---|
| `moray_east_wind_203501.nc`, `tico_wind_203501.nc` | `u`, `v` @ 100m, hourly, point | Wind |
| `tico_wind_203507.nc` | `u`, `v` @ 100m, hourly, point (summer) | Solar (Fuendetodos, summer) |
| `zaragoza_temp_203501.nc` | `2t`, hourly, point | Demand |
| `fuendetodos_solar_203501.nc`, `fuendetodos_solar_203507.nc` | `2t`, `avg_sdswrf`, point, winter+summer | Solar |
| `global_2t_monthly_203501.nc` | `2t`, monthly mean, global | Basic (global evidence) |
| `europe_2t_203501.nc` | `2t`, single timestep, European crop | Context maps |

`data/_wind_*.nc` and `data/_sfc_*.nc` are per-chunk intermediate
downloads, kept so an interrupted retrieval only needs to redo the
current chunk. Safe to delete once the per-site files above are
confirmed correct.
